<h3 style="color: #777777;"> Paso 0: Validar la ubicación de la base de datos</h3>

In [1]:
# MINIMAL DEBUG SCRIPT
import sqlite3
from pathlib import Path
import pandas as pd

print("1. Current directory:", Path.cwd())
print("\n2. Looking for nursery.db...")

# Try exact paths that should work
paths_to_try = [
    '../nursery.db',  # Most likely if you're in 'reports' folder
    'nursery.db',
    '../../nursery.db',
]

for path_str in paths_to_try:
    path = Path(path_str)
    print(f"\nTrying: {path_str}")
    print(f"  Absolute: {path.absolute()}")
    print(f"  Exists: {path.exists()}")
    
    if path.exists():
        try:
            conn = sqlite3.connect(str(path))
            cursor = conn.cursor()
            
            # List ALL tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
            tables = [t[0] for t in cursor.fetchall()]
            
            print(f"  ✅ Connected! Tables: {tables}")
            
            # If OBSERVACION_LOTE exists, show sample
            if 'OBSERVACION_LOTE' in tables:
                cursor.execute("SELECT * FROM OBSERVACION_LOTE LIMIT 3")
                sample = cursor.fetchall()
                print(f"  Sample from OBSERVACION_LOTE: {sample}")
            else:
                # Check for any observation-like table
                for table in tables:
                    if 'OBSERV' in table.upper():
                        cursor.execute(f"SELECT * FROM {table} LIMIT 3")
                        sample = cursor.fetchall()
                        print(f"  Sample from {table}: {sample}")
            
            conn.close()
            break
            
        except Exception as e:
            print(f"  ❌ Error: {e}")

1. Current directory: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager

2. Looking for nursery.db...

Trying: ../nursery.db
  Absolute: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager/../nursery.db
  Exists: False

Trying: nursery.db
  Absolute: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager/nursery.db
  Exists: True
  ✅ Connected! Tables: ['sqlite_sequence', 'GASTOS', 'REGISTRO_CLIMA', 'ENTRADA_PLANTAS', 'CATEGORIA', 'ETAPA_CRECIMIENTO', 'OBSERVACION_LOTE', 'REGISTRO_RIEGO', 'METODOS_PROPAGACION', 'ESPECIE_LOTE', 'CONTROL_PLAGAS', 'OBSERVACION_LOTE_BACKUP_12_19_25']
  Sample from OBSERVACION_LOTE: [(1, 6, '2025-09-22', 76, 20.0, 'BOLSA_PEQUEÑA', 'Buena', 'Requerimiento control de plagas y división de 6 papayos'), (2, 4, '2025-09-23', 39, 15.0, 'BOLSA_PEQUEÑA', 'Buena', 'Tiene algunas hojas deformes'), (3, 3, '2025-09-24', 88, 33.0, 'PRE-EMBOLSE', 'Buena', 'Algunas hojas con huecos y bordes café y

<h3 style="color: #777777;"> Paso 1: Seleccionar los datos para el reporte</h3>

In [14]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import HTML, display

def get_observation_data(start_date=None, end_date=None, db_path='nursery.db'):
    """
    Read observation data from the OBSERVACION_LOTE table with optional date filtering.
    """
    # Convert to Path object and resolve relative path
    db_file = Path(db_path)
    
    # Check if database exists
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    
    try:
        # Connect to database 
        conn = sqlite3.connect(str(db_file))

        # Build the query with optional date filtering
        query = """ 
        SELECT * FROM OBSERVACION_LOTE_BACKUP_12_19_25
        WHERE 1=1 -- Always true makes it easier to add conditions
        """

        # Add date filters if provided
        params = []
        if start_date:
            query += " AND FECHA_OBSERVACION >= ?"
            params.append(start_date)
        if end_date:
            query += " AND FECHA_OBSERVACION <= ?"
            params.append(end_date)

        # Order by date (most recent first)
        query += " ORDER BY FECHA_OBSERVACION DESC"

        # Execute the query and get data
        df = pd.read_sql_query(query, conn, params=params if params else None)
        
        return df
    
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

def generate_html_report(df, title="VIVERO RAÍCES DORADAS - REPORTE OPERATIVO"):
    """
    Generate an HTML report from observation data.
    
    Args:
        df (pandas.DataFrame): Observation data
        title (str): Report title
    
    Returns:
        str: HTML string
    """
    if df.empty:
        return "<h3>No data available</h3>"
    
    # Calculate statistics
    total_observations = len(df)
    date_range = f"{df['FECHA_OBSERVACION'].min()} a {df['FECHA_OBSERVACION'].max()}"
    unique_lots = df['ESPECIE_LOTE_ID'].nunique()
    available_columns = len(df.columns)
    
    # Create the HTML
    html_content = f'''
    <center><h1 style="color: #777777;">{title}</h1></center>
    <center><h2 style="color: #777777;">Observación de Plantas</h2></center>
    
    <div align="center">
        <table style="width: 80%; border-collapse: collapse; margin: 20px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Rango de Fechas</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Conteo de Observaciones</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Lotes Existentes</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Columnas disponibles</th>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{date_range}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_observations:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{unique_lots}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{available_columns}</td>
            </tr>
        </table>
    </div>
    
    <div align="center" style="margin: 20px;">
        <h3 style="color: #777777;">Resumen histórico de datos</h3>
        <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Nombre de Columna</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Únicos</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Totales</th>
            </tr>
    '''
    
    # Add row for each column
    for column in df.columns:
        unique_count = df[column].nunique()
        value_count = df[column].shape[0]
        
        html_content += f'''
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{column}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{unique_count}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{value_count}</td>
            </tr>
        '''
    
    # Close the column detail table
    html_content += '''
        </table>
    </div>
    
    <div align="center" style="margin: 30px;">
    <center><h2 style="color: #777777;">Análisis de Inventario</h2></center>
        <h3 style="color: #777777;">Conteo por Etapa de Crecimiento</h3>
        <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Etapa de Crecimiento</th>
                <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Inventario Actual</th>
                <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Porcentaje</th>
                
            </tr>
    '''
    
    # Add growth stage summary
    if 'ETAPA_CRECIMIENTO' in df.columns and 'CANTIDAD_PLANTAS_VIVAS' in df.columns:
        # Get the latest date in the dataset
        latest_date = df['FECHA_OBSERVACION'].max()
        
        # Get current inventory (most recent observation per lot)
        # Sort by date descending, then get first record per lot
        df_sorted = df.sort_values('FECHA_OBSERVACION', ascending=False)
        current_inventory = df_sorted.drop_duplicates(subset='ESPECIE_LOTE_ID', keep='first')
        
        # Calculate current plant count by stage (sum of CANTIDAD_PLANTAS_VIVAS)
        current_plant_counts = current_inventory.groupby('ETAPA_CRECIMIENTO')['CANTIDAD_PLANTAS_VIVAS'].sum()
        
        # Get total plants for percentage calculation
        total_current_plants = current_inventory['CANTIDAD_PLANTAS_VIVAS'].sum()
        # Sort descending plant count for readability
        current_plant_counts = current_plant_counts.sort_values(ascending=False)

        
        for stage, plant_count in current_plant_counts.items():
            # Calculate percentage based on current inventory
            percentage = (plant_count / total_current_plants) * 100 if total_current_plants > 0 else 0
            
            
            html_content += f'''
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{stage}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{plant_count:,}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{percentage:.1f}%</td>
                </tr>
            '''
        
        # Add a summary row
        html_content += f'''
            <tr style="background-color: #f5f5f5;">
                <td style="padding: 10px; border: 1px solid #ddd; color: #333; font-weight: bold;">TOTAL</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{total_current_plants:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #666; font-weight: bold;">100.0%</td>
            </tr>
        '''
        
        # Add a note about the current inventory date
        html_content += f'''
        </table>
        <p style="color: #777777; font-size: 12px; margin-top: 10px;">
            <em>Inventario actual basado en la última observación (Fecha más reciente: {latest_date})</em>
        </p>
    '''
    else:
        html_content += '''
            <tr>
                <td colspan="4" style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #666;">
                    No hay datos de etapa de crecimiento o cantidad de plantas disponibles
                </td>
            </tr>
        </table>
    '''
    
    # Add date for footer
    from datetime import datetime
    current_date = datetime.now().strftime('%Y-%m-%d')
    
    html_content += f'''
    </div>
    
    <div align="center" style="margin-top: 40px; padding: 20px; background-color: #f9f9f9; border-top: 2px solid #e3d0b3;">
        <p style="color: #777777; font-size: 12px;">
            Reporte generado automáticamente • Vivero Raíces Doradas • {current_date}
        </p>
    </div>
    '''
    
    return html_content


def display_report(df=None, start_date=None, end_date=None, db_path='nursery.db'):
    """
    Complete function to get data and display HTML report.
    
    Args:
        df (pandas.DataFrame, optional): Pre-loaded data. If None, loads from database.
        start_date (str, optional): Start date for filtering
        end_date (str, optional): End date for filtering
        db_path (str, optional): Database path
    """
    from datetime import datetime
    
    # Get data if not provided
    if df is None:
        print("📊 Cargando datos de la base de datos...")
        df = get_observation_data(start_date=start_date, end_date=end_date, db_path=db_path)
        print(f"✅ Datos cargados: {len(df)} registros")
    
    # Generate and display HTML report
    html_report = generate_html_report(df)
    display(HTML(html_report) )
    
    # Return the dataframe for further analysis
    return df


# Usage examples in Jupyter
if __name__ == "__main__":
    #print("Iniciando reporte del vivero...\n")
    
    # Option 1: Display full report
    #data = display_report()
    
    #print("\n" + "="*80 + "\n")
    
    # Option 2: Display filtered report
    print("Generando reporte filtrado...")
    filtered_data = display_report(start_date='2025-10-27')
    

Generando reporte filtrado...
📊 Cargando datos de la base de datos...
✅ Datos cargados: 825 registros


In [15]:
import sqlite3
import pandas as pd 
from pathlib import Path
from typing import Optional, List, Dict, Any
from IPython.display import HTML, display

def get_left_table_data(
    db_path: str = 'nursery.db',
    table_name: str = 'ESPECIE_LOTE',
    columns: Optional[List[str]] = None,
    left_join_key: str = 'ID',
    rename_key_for_join: Optional[str] = None
) -> pd.DataFrame:
    """
    Read data from a dimension table for left join operations.
    Provides traceability and displays shape overview in Jupyter.
    
    Args:
        db_path: Path to SQLite database
        table_name: Name of the left/dimension table
        columns: Specific columns to select (None = all columns)
        left_join_key: Primary key column name in the left table
        rename_key_for_join: Optional new name for the key column 
                            (useful when left table key has different name than right table foreign key)
        
    Returns:
        DataFrame with left table data ready for merging
    """
    print(f"📊 STEP 1: LOADING LEFT TABLE DATA")
    print(f"   Table: {table_name}")
    print(f"   Key column: {left_join_key}")
    
    db_file = Path(db_path)
    
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    
    try:
        conn = sqlite3.connect(str(db_file))
        
        # Get table info for validation
        table_info = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
        available_columns = table_info['name'].tolist()
        
        print(f"   Available columns in {table_name}:")
        for col in available_columns:
            print(f"     - {col}")
        
        # Validate key column exists
        if left_join_key not in available_columns:
            raise ValueError(f"Key column '{left_join_key}' not found in table '{table_name}'")
        
        # Build column selection - always include the key column
        if columns:
            # Ensure key column is included
            if left_join_key not in columns:
                columns = [left_join_key] + columns
                print(f"   🔑 Added key column '{left_join_key}' to selection")
            
            # Validate all requested columns exist
            missing_cols = [col for col in columns if col not in available_columns]
            if missing_cols:
                raise ValueError(f"Columns not found in table '{table_name}': {missing_cols}")
            
            column_str = ', '.join(columns)
            selected_columns = columns
        else:
            column_str = '*'
            selected_columns = available_columns
        
        # Build and execute query
        query = f"""
        SELECT {column_str} 
        FROM {table_name}
        """
        
        print(f"\n   Executing query:")
        print(f"   {query}\n")
        
        df = pd.read_sql_query(query, conn)
        
        # Rename key column if specified
        if rename_key_for_join and rename_key_for_join != left_join_key:
            df = df.rename(columns={left_join_key: rename_key_for_join})
            print(f"   🔄 Renamed key column from '{left_join_key}' to '{rename_key_for_join}'")
        
        # Display shape overview
        print(f"✅ DATA LOADED SUCCESSFULLY")
        print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"   Columns selected: {', '.join(df.columns.tolist())}")
        
        # Display first few rows in HTML format
        display_data_overview(df, table_name)
        
        return df
        
    except sqlite3.Error as e:
        print(f"❌ Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

def display_data_overview(df: pd.DataFrame, table_name: str, max_rows: int = 5) -> None:
    """
    Display DataFrame overview in Jupyter with HTML formatting.
    """
    print(f"\n📋 {table_name} DATA PREVIEW:")
    
    if df.empty:
        print("   No data available")
        return
    
    # Create styled HTML table
    styled_df = df.head(max_rows).style.set_table_styles([
        {'selector': 'thead th', 'props': [('background-color', '#f0f0f0'), 
                                         ('font-weight', 'bold'),
                                         ('text-align', 'left'),
                                         ('padding', '8px'),
                                         ('border', '1px solid #ddd')]},
        {'selector': 'td', 'props': [('padding', '8px'),
                                    ('border', '1px solid #ddd')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]}
    ]).set_caption(f"First {min(max_rows, len(df))} rows of {table_name}")
    
    display(styled_df)
    
    # Show data types
    print(f"\n📝 COLUMN DATA TYPES:")
    dtypes_df = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [str(dtype) for dtype in df.dtypes],
        'Non-Null Count': df.count().values
    })
    display(HTML(dtypes_df.to_html(index=False, classes='table table-sm')))
    
    # Show unique counts for key columns
    if len(df.columns) > 0:
        print(f"\n🔑 UNIQUE VALUES IN KEY COLUMNS:")
        key_info = []
        for col in df.columns[:3]:  # Show first 3 columns
            unique_count = df[col].nunique()
            key_info.append({
                'Column': col,
                'Unique Values': unique_count,
                'Duplicates': 'No' if len(df) == unique_count else 'Yes'
            })
        display(HTML(pd.DataFrame(key_info).to_html(index=False, classes='table table-sm')))

        # Cell 1: Test with ESPECIE_LOTE table
print("="*60)
print("TEST 1: Loading ESPECIE_LOTE (core species table)")
print("="*60)

especie_df = get_left_table_data(
    db_path='nursery.db',
    table_name='ESPECIE_LOTE',
    columns=['ID', 'NOMBRE_COMUN', 'NOMBRE_CIENTIFICO', 'CATEGORIA', 'ESTADO'],
    left_join_key='ID',
    rename_key_for_join=None  # Keep as 'ID' since it matches right table's ESPECIE_LOTE_ID
)

print(f"\n💾 DataFrame ready for next stage: 'especie_df'")
print(f"   Use this variable in the next cell for merging")


TEST 1: Loading ESPECIE_LOTE (core species table)
📊 STEP 1: LOADING LEFT TABLE DATA
   Table: ESPECIE_LOTE
   Key column: ID
   Available columns in ESPECIE_LOTE:
     - ID
     - NOMBRE_CIENTIFICO
     - NOMBRE_COMUN
     - FAMILIA
     - FECHA_INGRESO
     - PROPAGACION
     - CATEGORIA_ID
     - ULTIMA_MODIFICACION


ValueError: Columns not found in table 'ESPECIE_LOTE': ['CATEGORIA', 'ESTADO']